# Bluetongue Virus Transmission Dynamics: Ruminant–Culicoides Model

**Deterministic 11-compartment host–vector mathematical model**

This notebook documents the Bluetongue virus (BTV) model in the supplied Colab file. It represents infection dynamics in small ruminants, cattle and *Culicoides* vectors.

## Contents
1. Model structure and assumptions
2. Symbolic formulation and next-generation matrix
3. Parameter values and initial conditions
4. Numerical model and basic reproduction number
5. Disease-free equilibrium and local stability
6. Numerical simulations
7. Normalized local sensitivity analysis
8. LHS–PRCC global sensitivity analysis

**Important scope note:** The source notebook listed temperature (`T`) and rainfall (`r`) values in its parameter dictionary, but these were not used in the numerical differential equations. The cleaned notebook does not claim that the current numerical model is climate-driven. Environmental forcing should be incorporated only after its mathematical form and parameterisation have been specified.


## 1. Model structure

The 11 compartments are:

| Population | Compartments |
|---|---|
| Small ruminants | `S_SR`, `E_SR`, `I_SR`, `R_SR` |
| Cattle | `S_C`, `E_C`, `I_C`, `R_C` |
| *Culicoides* vectors | `S_V`, `E_V`, `I_V` |

The infected-state ordering used in the next-generation matrix is \((E_{SR}, I_{SR}, E_C, I_C, E_V, I_V)^T\). The host population in the supplied force-of-infection expressions excludes vectors.

The numerical force of infection is retained from the uploaded notebook:
\[
\lambda_{SR}=\frac{\beta_{SR}aI_V}{N_H},\quad
\lambda_C=\frac{\beta_CaI_V}{N_H},\quad
\lambda_V=\frac{\beta_Va(I_{SR}+I_C)}{N_H}.
\]


In [ ]:
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.integrate import solve_ivp
from scipy.linalg import eigvals, lstsq
from scipy.stats import qmc, rankdata, t as student_t

sp.init_printing()


## 2. Symbolic model and next-generation matrix

The symbolic formulation below follows the infected-state ordering above. \(\mathcal F\) contains new infection terms, while \(\mathcal V\) contains all other transfers. The next-generation matrix is \(K=FV^{-1}\), evaluated at the disease-free equilibrium.


In [ ]:

import sympy as sp
from IPython.display import display

sp.init_printing()

# ============================================================
# 1. STATE VARIABLES: 11 COMPARTMENTS
# ============================================================

S_SR, E_SR, I_SR, R_SR = sp.symbols(
    'S_SR E_SR I_SR R_SR', nonnegative=True
)

S_C, E_C, I_C, R_C = sp.symbols(
    'S_C E_C I_C R_C', nonnegative=True
)

S_V, E_V, I_V = sp.symbols(
    'S_V E_V I_V', nonnegative=True
)

X = sp.Matrix([
    S_SR, E_SR, I_SR, R_SR,
    S_C, E_C, I_C, R_C,
    S_V, E_V, I_V
])

# ============================================================
# 2. PARAMETERS
# ============================================================

Lambda_SR, Lambda_C, Lambda_V = sp.symbols(
    'Lambda_SR Lambda_C Lambda_V', positive=True
)

beta_SR, beta_C, beta_V = sp.symbols(
    'beta_SR beta_C beta_V', positive=True
)

mu_SR, mu_C, mu_V = sp.symbols(
    'mu_SR mu_C mu_V', positive=True
)

delta_SR, delta_C, rho_V = sp.symbols(
    'delta_SR delta_C rho_V', positive=True
)

gamma_SR, gamma_C = sp.symbols(
    'gamma_SR gamma_C', positive=True
)

sigma_SR, sigma_C = sp.symbols(
    'sigma_SR sigma_C', nonnegative=True
)

a = sp.symbols('a', positive=True)

# ============================================================
# 3. TOTAL HOST POPULATION
# VECTORS ARE NOT INCLUDED IN N_H
# ============================================================

N_H = (
    S_SR + E_SR + I_SR + R_SR
    + S_C + E_C + I_C + R_C
)

# ============================================================
# 4. FORCE OF INFECTION
# ============================================================

lambda_SR = beta_SR * a * I_V / N_H
lambda_C = beta_C * a * I_V / N_H

lambda_V = beta_V * a * (I_SR + I_C) / N_H

# ============================================================
# 5. FULL ODE SYSTEM: ALL 11 EQUATIONS
# ============================================================

f = sp.Matrix([

    # Small ruminants
    Lambda_SR - lambda_SR*S_SR - mu_SR*S_SR,

    lambda_SR*S_SR - (delta_SR + mu_SR)*E_SR,

    delta_SR*E_SR
    - (gamma_SR + sigma_SR + mu_SR)*I_SR,

    gamma_SR*I_SR - mu_SR*R_SR,

    # Cattle
    Lambda_C - lambda_C*S_C - mu_C*S_C,

    lambda_C*S_C - (delta_C + mu_C)*E_C,

    delta_C*E_C
    - (gamma_C + sigma_C + mu_C)*I_C,

    gamma_C*I_C - mu_C*R_C,

    # Culicoides
    Lambda_V - lambda_V*S_V - mu_V*S_V,

    lambda_V*S_V - (rho_V + mu_V)*E_V,

    rho_V*E_V - mu_V*I_V

])

print("FULL ODE SYSTEM")
for variable, equation in zip(X, f):
    display(sp.Eq(sp.Derivative(variable, sp.Symbol('t')),
                  equation))

# ============================================================
# 6. INFECTED-STATE VECTOR
# ORDER: E_SR, I_SR, E_C, I_C, E_V, I_V
# ============================================================

Y = sp.Matrix([E_SR, I_SR, E_C, I_C, E_V, I_V])

# ============================================================
# 7. F VECTOR: ONLY NEW INFECTIONS
# ============================================================

F_vec = sp.Matrix([
    lambda_SR*S_SR,  # New infections in small ruminants
    0,
    lambda_C*S_C,    # New infections in cattle
    0,
    lambda_V*S_V,    # New infections in vectors
    0
])

# ============================================================
# 8. V VECTOR: OTHER TRANSFERS
# CONVENTION: V = OUTFLOW - INFLOW
# ============================================================

V_vec = sp.Matrix([
    (delta_SR + mu_SR)*E_SR,

    (gamma_SR + sigma_SR + mu_SR)*I_SR
    - delta_SR*E_SR,

    (delta_C + mu_C)*E_C,

    (gamma_C + sigma_C + mu_C)*I_C
    - delta_C*E_C,

    (rho_V + mu_V)*E_V,

    mu_V*I_V - rho_V*E_V
])

# ============================================================
# 9. NEXT-GENERATION MATRICES
# ============================================================

F = F_vec.jacobian(Y)
V = V_vec.jacobian(Y)

print("F MATRIX")
display(F)

print("V MATRIX")
display(V)

# ============================================================
# 10. DISEASE-FREE EQUILIBRIUM (DFE)
# ============================================================

dfe = {
    S_SR: Lambda_SR / mu_SR,
    E_SR: 0,
    I_SR: 0,
    R_SR: 0,

    S_C: Lambda_C / mu_C,
    E_C: 0,
    I_C: 0,
    R_C: 0,

    S_V: Lambda_V / mu_V,
    E_V: 0,
    I_V: 0
}

print("DISEASE-FREE EQUILIBRIUM")
display(dfe)

# ============================================================
# 11. SUBSTITUTE DFE INTO F AND V
# ============================================================

F0 = sp.simplify(F.subs(dfe))
V0 = sp.simplify(V.subs(dfe))

print("F AT DFE")
display(F0)

print("V AT DFE")
display(V0)

# ============================================================
# 12. NEXT-GENERATION MATRIX K = F0 * INVERSE(V0)
# ============================================================

K = sp.simplify(F0 * V0.inv())

print("NEXT-GENERATION MATRIX K = F0 * V0^(-1)")
display(K)

# ============================================================
# 13. SYMBOLIC BASIC REPRODUCTION NUMBER R0
# ============================================================

N_H0 = sp.simplify(
    Lambda_SR / mu_SR + Lambda_C / mu_C
)

R0_squared = sp.factor(
    (
        beta_SR * a * Lambda_SR / (mu_SR * N_H0)
        * delta_SR / (delta_SR + mu_SR)
        / (gamma_SR + sigma_SR + mu_SR)
        * beta_V * a * Lambda_V / (mu_V * N_H0)
        * rho_V / (rho_V + mu_V)
        / mu_V
    )
    +
    (
        beta_C * a * Lambda_C / (mu_C * N_H0)
        * delta_C / (delta_C + mu_C)
        / (gamma_C + sigma_C + mu_C)
        * beta_V * a * Lambda_V / (mu_V * N_H0)
        * rho_V / (rho_V + mu_V)
        / mu_V
    )
)

R0 = sp.sqrt(R0_squared)

print("R0 SQUARED")
display(sp.Eq(sp.Symbol('R_0_squared'), R0_squared))

print("BASIC REPRODUCTION NUMBER R0")
display(sp.Eq(sp.Symbol('R_0'), R0))

# ============================================================
# 14. VERIFY THAT THE MATRIX SPECTRAL RADIUS IS R0
# ============================================================

print("NONZERO EIGENVALUES OF K")
display(sp.factor(K.charpoly().as_expr()))

print("R0 FORMULA DERIVED FROM THE TWO HOST-VECTOR PATHWAYS")
display(sp.Eq(sp.Symbol('R_0'), R0))


## 3. Parameter values and initial conditions

The following values are transcribed from the uploaded notebook. They are retained as supplied and have not been independently validated against published sources. The model uses consistent time units; the original simulation labels time in days.


In [ ]:
# ============================================================
# 1. PARAMETER VALUES
# ============================================================

p = {
    "Lambda_SR": 20.0,
    "Lambda_C": 10.0,
    "Lambda_V": 850.0,

    "mu_SR": 0.0005,
    "mu_C": 0.0003,
    "mu_V": 0.05,

    "sigma_SR": 0.05,
    "sigma_C": 0.005,

    "beta_SR": 0.15,
    "beta_C": 0.10,
    "beta_V": 0.008,

    "a": 0.18,

    "delta_SR": 0.14,
    "gamma_SR": 0.07,

    "delta_C": 0.10,
    "gamma_C": 0.05,

    "rho_V": 0.10,

    # Temperature and rainfall were listed in the source notebook,
    # but are not used in the supplied ODE equations.
}

# ============================================================
# 2. INITIAL CONDITIONS FROM YOUR TABLE
# ============================================================

y0 = np.array([
    39985, 10, 5, 1,
    33325, 5, 3, 1,
    16975, 15, 10
], dtype=float)

names = [
    "S_SR", "E_SR", "I_SR", "R_SR",
    "S_C", "E_C", "I_C", "R_C",
    "S_V", "E_V", "I_V"
]

print("\nINITIAL CONDITIONS")
display(pd.DataFrame({
    "Compartment": names,
    "Initial value": y0
}))

print("\nPARAMETERS")
display(pd.DataFrame({
    "Parameter": list(p.keys()),
    "Value": list(p.values())
}))



## 4. Numerical ODE model, reproduction number, DFE and stability

The following cell contains the 11-compartment numerical model and the calculations retained from the supplied notebook. A central finite-difference Jacobian helper is used instead of a private SciPy internal function.


In [ ]:
def approx_derivative(fun, x, method="3-point"):
    """Central finite-difference Jacobian for a vector-valued function."""
    x = np.asarray(x, dtype=float)
    f0 = np.asarray(fun(x), dtype=float)
    J = np.zeros((f0.size, x.size), dtype=float)
    for j in range(x.size):
        h = 1e-6 * max(1.0, abs(x[j]))
        xp, xm = x.copy(), x.copy()
        xp[j] += h
        xm[j] -= h
        J[:, j] = (np.asarray(fun(xp)) - np.asarray(fun(xm))) / (2*h)
    return J

# ============================================================
# 3. ODE MODEL
# ============================================================

def model(t, y, p):

    S_SR, E_SR, I_SR, R_SR, \
    S_C, E_C, I_C, R_C, \
    S_V, E_V, I_V = y

    N_H = (
        S_SR + E_SR + I_SR + R_SR
        + S_C + E_C + I_C + R_C
    )

    # Forces of infection
    lambda_SR = p["beta_SR"] * p["a"] * I_V / N_H
    lambda_C  = p["beta_C"] * p["a"] * I_V / N_H
    lambda_V  = p["beta_V"] * p["a"] * (
        I_SR + I_C
    ) / N_H

    # Small ruminants
    dS_SR = (
        p["Lambda_SR"] - lambda_SR*S_SR
        - p["mu_SR"]*S_SR
    )

    dE_SR = (
        lambda_SR*S_SR
        - (p["delta_SR"] + p["mu_SR"])*E_SR
    )

    dI_SR = (
        p["delta_SR"]*E_SR
        - (p["gamma_SR"] + p["sigma_SR"]
           + p["mu_SR"])*I_SR
    )

    dR_SR = p["gamma_SR"]*I_SR - p["mu_SR"]*R_SR

    # Cattle
    dS_C = (
        p["Lambda_C"] - lambda_C*S_C
        - p["mu_C"]*S_C
    )

    dE_C = (
        lambda_C*S_C
        - (p["delta_C"] + p["mu_C"])*E_C
    )

    dI_C = (
        p["delta_C"]*E_C
        - (p["gamma_C"] + p["sigma_C"]
           + p["mu_C"])*I_C
    )

    dR_C = p["gamma_C"]*I_C - p["mu_C"]*R_C

    # Culicoides
    dS_V = (
        p["Lambda_V"] - lambda_V*S_V
        - p["mu_V"]*S_V
    )

    dE_V = (
        lambda_V*S_V
        - (p["rho_V"] + p["mu_V"])*E_V
    )

    dI_V = p["rho_V"]*E_V - p["mu_V"]*I_V

    return np.array([
        dS_SR, dE_SR, dI_SR, dR_SR,
        dS_C, dE_C, dI_C, dR_C,
        dS_V, dE_V, dI_V
    ])

# ============================================================
# 4. BASIC REPRODUCTION NUMBER R0
# ============================================================

def calculate_R0(p):

    S_SR0 = p["Lambda_SR"] / p["mu_SR"]
    S_C0  = p["Lambda_C"] / p["mu_C"]
    S_V0  = p["Lambda_V"] / p["mu_V"]

    N_H0 = S_SR0 + S_C0

    A_SR = (
        p["beta_SR"] * S_SR0 * p["delta_SR"]
        / (
            (p["delta_SR"] + p["mu_SR"])
            * (p["gamma_SR"] + p["sigma_SR"]
               + p["mu_SR"])
        )
    )

    A_C = (
        p["beta_C"] * S_C0 * p["delta_C"]
        / (
            (p["delta_C"] + p["mu_C"])
            * (p["gamma_C"] + p["sigma_C"]
               + p["mu_C"])
        )
    )

    R0_squared = (
        (p["a"] / N_H0)**2
        * p["beta_V"] * S_V0
        * p["rho_V"]
        / (
            p["mu_V"]**2
            * (p["rho_V"] + p["mu_V"])
        )
        * (A_SR + A_C)
    )

    return np.sqrt(R0_squared)

R0 = calculate_R0(p)

print("\nBASIC REPRODUCTION NUMBER")
print(f"R0 = {R0:.8f}")

if R0 < 1:
    print("R0 < 1: infection cannot invade near the DFE.")
else:
    print("R0 > 1: infection may invade near the DFE.")

# ============================================================
# 5. DISEASE-FREE EQUILIBRIUM
# ============================================================

dfe = np.array([
    p["Lambda_SR"]/p["mu_SR"], 0, 0, 0,
    p["Lambda_C"]/p["mu_C"], 0, 0, 0,
    p["Lambda_V"]/p["mu_V"], 0, 0
], dtype=float)

print("\nDISEASE-FREE EQUILIBRIUM")
display(pd.DataFrame({
    "Compartment": names,
    "DFE": dfe
}))

# ============================================================
# 6. JACOBIAN AND LOCAL STABILITY
# ============================================================

J = approx_derivative(
    lambda y: model(0, y, p),
    dfe,
    method="3-point"
)

eigenvalues = eigvals(J)

print("\nJACOBIAN EIGENVALUES AT THE DFE")
display(pd.DataFrame({
    "Real part": np.real(eigenvalues),
    "Imaginary part": np.imag(eigenvalues)
}))

if np.all(np.real(eigenvalues) < 0):
    print("DFE is locally asymptotically stable.")
else:
    print("DFE is not locally asymptotically stable.")

# ============================================================
# 7. NUMERICAL SIMULATION
# ============================================================

days = 365

sol = solve_ivp(
    lambda t, y: model(t, y, p),
    (0, days),
    y0,
    method="LSODA",
    dense_output=True,
    rtol=1e-8,
    atol=1e-8
)

if not sol.success:
    raise RuntimeError(sol.message)

if np.min(sol.y) < -1e-5:
    print("Warning: negative compartment values detected.")

# Plot all compartments
plt.figure(figsize=(12, 6))

for i, name in enumerate(names):
    plt.plot(sol.t, sol.y[i], label=name)

plt.xlabel("Time (days)")
plt.ylabel("Population")
plt.title("Bluetongue transmission dynamics")
plt.legend(ncol=3, fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Separate plot of infected populations
plt.figure(figsize=(10, 5))

for name in ["I_SR", "I_C", "I_V"]:
    idx = names.index(name)
    plt.plot(sol.t, sol.y[idx], label=name)

plt.xlabel("Time (days)")
plt.ylabel("Infected population")
plt.title("Infected small ruminants, cattle and vectors")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# 8. LOCAL NORMALIZED SENSITIVITY OF R0
# ============================================================

sensitivity_parameters = [
    "beta_SR", "beta_C", "beta_V", "a",
    "delta_SR", "delta_C",
    "gamma_SR", "gamma_C",
    "sigma_SR", "sigma_C",
    "mu_SR", "mu_C", "mu_V",
    "rho_V", "Lambda_SR", "Lambda_C", "Lambda_V"
]

def normalized_sensitivity(p, key, h=1e-5):

    p_plus = p.copy()
    p_minus = p.copy()

    p_plus[key] = p[key] * (1 + h)
    p_minus[key] = p[key] * (1 - h)

    R_plus = calculate_R0(p_plus)
    R_minus = calculate_R0(p_minus)

    derivative = (R_plus - R_minus) / (
        2 * h * p[key]
    )

    return p[key] * derivative / calculate_R0(p)

sens = []

for key in sensitivity_parameters:
    sens.append({
        "Parameter": key,
        "Normalized sensitivity": normalized_sensitivity(
            p, key
        )
    })

sens_df = pd.DataFrame(sens)
sens_df["Absolute sensitivity"] = (
    sens_df["Normalized sensitivity"].abs()
)
sens_df = sens_df.sort_values(
    "Absolute sensitivity", ascending=False
)

print("\nLOCAL SENSITIVITY INDICES OF R0")
display(sens_df)

plt.figure(figsize=(10, 6))
plt.barh(
    sens_df["Parameter"],
    sens_df["Normalized sensitivity"]
)
plt.axvline(0, linewidth=1)
plt.xlabel("Normalized sensitivity index")
plt.title("Local sensitivity of R0")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()



## 5. Global sensitivity analysis: LHS–PRCC

The original notebook contained both placeholder parameter bounds and a separate synthetic test block. The synthetic test block has been removed. The function below requires literature-supported lower and upper bounds for every parameter before it can run.


In [ ]:
# Enter literature-supported lower and upper bounds for every parameter.
# None placeholders intentionally prevent accidental use of invented ranges.
parameter_bounds = {
    key: (None, None) for key in p.keys()
}

def partial_rank_correlation(X, y):
    X_rank = np.column_stack([rankdata(X[:, j]) for j in range(X.shape[1])])
    y_rank = rankdata(y)
    n_samples, n_parameters = X_rank.shape
    output = []

    for j in range(n_parameters):
        Z = np.column_stack([
            np.ones(n_samples),
            np.delete(X_rank, j, axis=1)
        ])
        residual_x = X_rank[:, j] - Z @ lstsq(Z, X_rank[:, j])[0]
        residual_y = y_rank - Z @ lstsq(Z, y_rank)[0]
        r = np.corrcoef(residual_x, residual_y)[0, 1]
        dof = n_samples - n_parameters - 1

        if dof > 0 and np.isfinite(r) and abs(r) < 1:
            t_stat = r * np.sqrt(dof / max(1e-15, 1 - r**2))
            p_value = 2 * student_t.sf(abs(t_stat), dof)
        else:
            p_value = np.nan

        output.append((r, p_value))

    return output


def lhs_prcc_analysis(p, bounds, n=5000, seed=2026):
    keys = list(p.keys())
    missing = [
        key for key in keys
        if key not in bounds
        or bounds[key][0] is None
        or bounds[key][1] is None
    ]
    if missing:
        raise ValueError(
            "Add literature-supported lower and upper bounds for: "
            + ", ".join(missing)
        )

    limits = np.asarray([bounds[key] for key in keys], dtype=float)
    if np.any(limits[:, 0] >= limits[:, 1]) or np.any(limits < 0):
        raise ValueError("Check the lower and upper bounds.")

    sampler = qmc.LatinHypercube(d=len(keys), seed=seed)
    sample_values = qmc.scale(
        sampler.random(n), limits[:, 0], limits[:, 1]
    )
    sampled = pd.DataFrame(sample_values, columns=keys)
    sampled["R0"] = [
        calculate_R0({**p, **row.to_dict()})
        for _, row in sampled[keys].iterrows()
    ]

    prcc = partial_rank_correlation(
        sampled[keys].to_numpy(), sampled["R0"].to_numpy()
    )
    summary = pd.DataFrame({
        "Parameter": keys,
        "PRCC": [item[0] for item in prcc],
        "p_value": [item[1] for item in prcc]
    })
    summary["Absolute_PRCC"] = summary["PRCC"].abs()
    summary = summary.sort_values("Absolute_PRCC", ascending=False)

    display(summary)
    plt.figure(figsize=(10, 7))
    plt.barh(summary["Parameter"], summary["PRCC"])
    plt.axvline(0, linewidth=1)
    plt.xlabel("Partial Rank Correlation Coefficient")
    plt.ylabel("Parameter")
    plt.title("Global Sensitivity of R0: LHS–PRCC")
    plt.grid(axis="x", alpha=0.3)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    return sampled, summary

# Run only after entering verified bounds:
# lhs_samples, prcc_results = lhs_prcc_analysis(p, parameter_bounds)


## 6. Reproducibility notes and limitations

- The compartment definitions, numerical parameter values and initial conditions are based on the supplied notebook.
- The source notebook contained an additional exploratory symbolic expression with temperature and rainfall terms. It was not consistent with the numerical ODE implementation and is not included in the main analysis.
- Temperature and rainfall are not active inputs in the cleaned numerical model.
- Parameter values and initial conditions have not been independently checked against literature or surveillance data.
- Global sensitivity analysis should only be interpreted after justified parameter bounds are supplied.
- The Jacobian eigenvalue calculation is a numerical local-stability check, not a complete analytical proof.
- The notebook starts with cleared outputs; run the cells in order to reproduce tables, calculations and plots.

### Recommended next steps
1. Verify all parameter definitions, units and references.
2. Confirm the biological interpretation of host-to-vector and vector-to-host transmission.
3. Define and justify temperature/rainfall forcing before describing the model as climate-dependent.
4. Add published parameter ranges for LHS–PRCC.
5. Validate simulated trajectories against suitable epidemiological or entomological observations, if data are available.
